## Training Model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score, auc, f1_score, recall_score, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import StratifiedKFold, cross_val_score

import xgboost
from lightgbm import LGBMClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor

from pathlib import Path
import sys
import os
import mlflow
import dagshub


PARENT_DIRECTORY = Path.cwd().parent
#print(Path.cwd())
#print(PARENT_DIRECTORY)
sys.path.append(str(PARENT_DIRECTORY))
# sys.path.append(f'{PARENT_DIRECTORY}/scripts')
#print(sys.path)

from scripts.eda import EDA

sns.set_theme(style= 'whitegrid', context= 'notebook', palette= 'deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Cargando Datos
with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)
MLFLOW = params['mlflow']
CONFIG = params['config']
TARGET = params['features']['target']
CATEGORICAL = params['features']['categorical']

train_fe = pd.read_parquet(CONFIG['data_train_fe'])
test_fe = pd.read_parquet(CONFIG['data_test_fe'])
oot_fe = pd.read_parquet(CONFIG['data_oot_fe'])
X_train_fe = train_fe.drop(TARGET, axis = 1)
y_train_fe = train_fe[TARGET]

In [3]:
## Mini inspección

# Missings
missings_count_col = [X_train_fe[d].isna().sum() for d in X_train_fe.columns]
missings_mean_col = [X_train_fe[d].isna().mean() for d in X_train_fe.columns]

missings_df = pd.DataFrame({
    'feature': X_train_fe.columns,
    'Cantidad de Missings': missings_count_col,
    '% de Missings': missings_mean_col
}).sort_values(by= 'Cantidad de Missings', ascending= False)
missings_df_mayor_cero = missings_df.loc[missings_df['Cantidad de Missings'] > 0]


In [4]:
## Definiendo las Features
FEATS_CATEGORICALS = CATEGORICAL
FEATS_RECENCIA = [
    feat for feat in X_train_fe.columns
    if feat.endswith('_recencia')
]
FEATS_NEGOCIO = [
    'tenure_months',
    'age',
    'monthly_charges'
]
FEATS_CALENDARIO = [
    'month_sin',
    'month_cos'
]
FEATS_TEMPORALES = [
    feat for feat in X_train_fe.columns
    if feat not in (FEATS_CATEGORICALS + FEATS_RECENCIA + FEATS_NEGOCIO + FEATS_CALENDARIO)
]

In [5]:
## Correlation (Pearson)

corr_pearson = X_train_fe.corr(method= 'pearson', numeric_only= True).abs()
mask_pearson = np.triu(
    np.ones_like(corr_pearson, dtype='bool'),
    k= 1 # diagonal no incluida
)
corr_pairs_pearson = (
    corr_pearson.where(mask_pearson)
    .stack() # vuelve las columnas -> Índice (devuelve serie multi-index), omite los NaN
    .reset_index() # convierte ese multi-index en columnas normales
)

corr_pairs_pearson.columns = [
    'feature_1',
    'feature_2',
    'correlacion_pearson'
]

corr_pairs_pearson = corr_pairs_pearson.sort_values(by= 'correlacion_pearson', ascending= False)
corr_pairs_pearson_above_0_85 = corr_pairs_pearson.query(
    'correlacion_pearson >= 0.85'
)

## Correlation (Spearman)

corr_spearman = X_train_fe.corr(method = 'spearman', numeric_only= True).abs()
mask_spearman = np.triu(
    np.ones_like(corr_spearman, dtype= 'bool'),
    k= 1
)
corr_pairs_spearman = (
    corr_spearman.where(mask_spearman)
    .stack()
    .reset_index()
)

corr_pairs_spearman.columns = [
    'feature_1',
    'feature_2',
    'correlacion_spearman'
]

corr_pairs_spearman = corr_pairs_spearman.sort_values(by= 'correlacion_spearman', ascending= False)
corr_pairs_spearman_above_0_85 = corr_pairs_spearman.query(
    'correlacion_spearman >= 0.85'
)

In [6]:
corr_pairs_pearson_above_0_85.head(20)

,feature_1,feature_2,correlacion_pearson
151,tenure_months,history_months,1.000000
19284,active_days_last_month_lag_1_trend_nom,active_days_last_month_lag_1_trend_ratio,0.995773
18519,usage_minutes_lag_1_trend_nom,usage_minutes_lag_1_trend_ratio,0.993860
13772,late_payments_lag_1_cumsum,late_payments_lag_1_sum_12,0.980338
19131,marketing_emails_opened_lag_1_trend_nom,marketing_emails_opened_lag_1_trend_ratio,0.970610
8435,late_payments_lag_1_std_6,late_payments_lag_1_range_6,0.968047
5203,complaints_last_3m_lag_1_mean_6,complaints_last_3m_lag_1_mean_12,0.968016
7977,complaints_last_3m_lag_1_std_6,complaints_last_3m_lag_1_range_6,0.967968
4744,support_calls_lag_1_mean_6,support_calls_lag_1_mean_12,0.966970
13924,late_payments_lag_1_frec_12,late_payments_lag_1_sum_12,0.962288


In [7]:
corr_pairs_spearman_above_0_85.head(20)

,feature_1,feature_2,correlacion_spearman
151,tenure_months,history_months,1.000000
19284,active_days_last_month_lag_1_trend_nom,active_days_last_month_lag_1_trend_ratio,0.999525
18519,usage_minutes_lag_1_trend_nom,usage_minutes_lag_1_trend_ratio,0.999335
13772,late_payments_lag_1_cumsum,late_payments_lag_1_sum_12,0.997888
19131,marketing_emails_opened_lag_1_trend_nom,marketing_emails_opened_lag_1_trend_ratio,0.996487
10456,active_days_last_month_lag_1_diff,active_days_last_month_lag_1_pct_change,0.994308
12854,support_calls_lag_1_cumsum,support_calls_lag_1_sum_12,0.993049
15760,usage_severity_sum_3m,usage_severity_mean_3m,0.992862
9691,usage_minutes_lag_1_diff,usage_minutes_lag_1_pct_change,0.992092
14690,marketing_emails_opened_lag_1_cumsum,marketing_emails_opened_lag_1_sum_12,0.989386


In [8]:
## Mutual Information

imputer_no_recencia = SimpleImputer(
    strategy= 'constant',
    fill_value= 0
)
imputer_recencia = SimpleImputer(
    strategy= 'constant',
    fill_value= 999
)

transformation_imputer = ColumnTransformer(
    transformers= [
        ('imputation_no_recencia', imputer_no_recencia, FEATS_TEMPORALES+FEATS_NEGOCIO),
        ('imputation_recencia', imputer_recencia, FEATS_RECENCIA)
    ],
    remainder='drop',
    verbose_feature_names_out= False
)

X_train_transform_mi = transformation_imputer.fit_transform(X= X_train_fe)
X_imp = pd.DataFrame(
    X_train_transform_mi,
    columns= transformation_imputer.get_feature_names_out(),
    index= X_train_fe.index
)

mi = mutual_info_classif(
    X= X_imp,
    y= y_train_fe,
    random_state= CONFIG['random_state']
)
mi_df = pd.DataFrame({
    'features': transformation_imputer.get_feature_names_out(),
    'mi': mi
}).sort_values(by= 'mi', ascending= False)

In [9]:
mi_df.head(20)

,features,mi
146,late_payments_lag_1_recencia,0.027924
148,usage_min_lag_1_recencia,0.019429
72,late_payments_lag_1_range_6,0.010475
73,late_payments_lag_1_range_12,0.010457
88,marketing_emails_opened_lag_1_frec_6,0.010318
71,complaints_last_3m_lag_1_range_12,0.010117
70,complaints_last_3m_lag_1_range_6,0.008847
69,support_calls_lag_1_range_12,0.008525
68,support_calls_lag_1_range_6,0.007676
100,active_days_last_month_freq_6m,0.007169


In [10]:
## VIF
vif_df = pd.DataFrame({
    'feature': X_imp.columns,
    'vif': [
        variance_inflation_factor(X_imp.values, d)
        for d in range(X_imp.shape[1])
    ]
}).sort_values(by= 'vif', ascending= False)


/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [11]:
vif_df.head(20)

,feature,vif
36,marketing_emails_opened_lag_1_mean_3,inf
24,usage_minutes_lag_1_mean_3,inf
27,support_calls_lag_1_mean_3,inf
29,support_calls_lag_1_mean_12,inf
111,usage_minutes_lag_1_trend_nom,inf
112,support_calls_lag_1_trend_nom,inf
30,complaints_last_3m_lag_1_mean_3,inf
113,complaints_last_3m_lag_1_trend_nom,inf
114,late_payments_lag_1_trend_nom,inf
115,marketing_emails_opened_lag_1_trend_nom,inf


In [12]:
## DROPS por alta relación lineal
# Se utilizará este dataset para entrenar la regresión logística en el escenario B

DROPS_COLS_1 = [
    'tenure_months'
]

DROPS_COLS_2 = [
    feat for feat in X_train_fe.columns
    if feat.endswith('_mean_3')
]

# Nuevas Feats Temporales
FEATS_TEMPORALES_NEW = [
    feat for feat in FEATS_TEMPORALES
    if feat not in DROPS_COLS_2 
]
FEATS_NEGOCIO_NEW = [
    feat for feat in FEATS_NEGOCIO
    if feat not in DROPS_COLS_1
]

X_train_fe_drop_logistic = X_train_fe.drop(DROPS_COLS_2+DROPS_COLS_1, axis= 1)

In [13]:
## Definiendo el Pipeline
pipe_categorical = Pipeline(
    steps= [
        ('simple_imputer', SimpleImputer(strategy= 'constant', fill_value= 'MISSING')),
        ('one_hot_encoder', OneHotEncoder(handle_unknown= 'ignore'))
    ]
)
pipe_numeric_temporales = Pipeline(
    steps= [
        ('simple_imputer', SimpleImputer(strategy='constant', fill_value= 0)),
        ('standar_scaler', StandardScaler())
    ]
)
pipe_numeric_recencia = Pipeline(
    steps= [
        ('simple_imputer', SimpleImputer(strategy='constant', fill_value= 999)),
        ('standar_scaler', StandardScaler())
    ]
)
pipe_numeric_negocio = Pipeline(
    steps= [
        ('simple_imputer', SimpleImputer(strategy='median')),
        ('standar_scaler', StandardScaler())
    ]
) # aqui añadiría missing indicator pero como no existen sigo con el proceso

## Flujo del modelo A (todas las features)
preprocessor_a = ColumnTransformer(
    transformers= [
        ('categoricals', pipe_categorical, FEATS_CATEGORICALS),
        ('numeric_temporales', pipe_numeric_temporales, FEATS_TEMPORALES),
        ('numeric_recencia', pipe_numeric_recencia, FEATS_RECENCIA),
        ('numeric_negocio', pipe_numeric_negocio, FEATS_NEGOCIO),
        ('numeric_calendario','passthrough', FEATS_CALENDARIO)
    ],
    remainder= 'drop',
    verbose_feature_names_out= False
)
X_train_preprocessed_a = preprocessor_a.fit_transform(X_train_fe)
features_names = (
    preprocessor_a.get_feature_names_out()
)

## Flujo del modelo B (todas las features menos las dropeadas por alta correlación primera instancia)
preprocessor_b = ColumnTransformer(
    transformers= [
        ('categoricals', pipe_categorical, FEATS_CATEGORICALS),
        ('numeric_temporales', pipe_numeric_temporales, FEATS_TEMPORALES_NEW),
        ('numeric_recencia', pipe_numeric_recencia, FEATS_RECENCIA),
        ('numeric_negocio', pipe_numeric_negocio, FEATS_NEGOCIO_NEW),
        ('numeric_calendario', 'passthrough', FEATS_CALENDARIO)
    ],
    remainder= 'drop',
    verbose_feature_names_out= False
)
X_train_preprocessed_b = preprocessor_b.fit_transform(X_train_fe_drop_logistic)
features_names = (
    preprocessor_b.get_feature_names_out()
)

Modelo A: Ingresan todas las variables

In [14]:
## Entrenamiento de Modelo base (Logistic Regression)
pipe_logistic_a = Pipeline(
    steps= [
        ('preprocessor', preprocessor_a),
        ('model', LogisticRegression(
            random_state= CONFIG['random_state'],
            max_iter= 1500,
            solver= 'lbfgs'
        ))
    ]
)

pipe_logistic_a.fit(X= X_train_fe, y= y_train_fe)
predict_proba_logistic = pipe_logistic_a.predict_proba(X= X_train_fe)[:,1]
roc_auc_logistic = roc_auc_score(y_true= y_train_fe, y_score= predict_proba_logistic)
roc_auc_logistic
# cv = StratifiedKFold(n_splits= 5, random_state= CONFIG['random_state'], shuffle= True)
# roc_score_logistic = cross_val_score(
#     X= X_train_fe,
#     y= y_train_fe,
#     estimator= pipe_logistic,
#     cv= cv,
#     scoring= 'roc_auc'
# )
# roc_score_logistic

0.5645068964893308

Ingresan todas las variables menos los DROPs planteados

In [15]:
## Entrenamiento de Modelo base (Logistic Regression)
pipe_logistic_b = Pipeline(
    steps= [
        ('preprocessor', preprocessor_b),
        ('model', LogisticRegression(
            random_state= CONFIG['random_state'],
            max_iter= 1500,
            solver= 'lbfgs'
        ))
    ]
)

pipe_logistic_b.fit(X= X_train_fe, y= y_train_fe)
predict_proba_logistic = pipe_logistic_b.predict_proba(X= X_train_fe)[:,1]
roc_auc_logistic = roc_auc_score(y_true= y_train_fe, y_score= predict_proba_logistic)
roc_auc_logistic
# cv = StratifiedKFold(n_splits= 5, random_state= CONFIG['random_state'], shuffle= True)
# roc_score_logistic = cross_val_score(
#     X= X_train_fe,
#     y= y_train_fe,
#     estimator= pipe_logistic,
#     cv= cv,
#     scoring= 'roc_auc'
# )
# roc_score_logistic

0.5644882190670839

Modelo 1: Solo Negocio

In [21]:
FEATS_FULL_MODEL_1 = FEATS_NEGOCIO
pipe_mod_2 = Pipeline(
    steps= [
        ('simple_imputer', SimpleImputer(strategy= 'constant', fill_value= 0)),
        ('standar_scaler', StandardScaler())
    ]
)
pipe_categorical = Pipeline(
    steps= [
        ('simple_imputer', SimpleImputer(strategy= 'constant', fill_value= 'MISSING')),
        ('one_hot_encoder', OneHotEncoder(handle_unknown= 'ignore'))
    ]
)

transform_mod_2 = ColumnTransformer(
    transformers= [
        ('feat_num', pipe_mod_2, FEATS_FULL_MODEL_1),
        ('feat_cat', pipe_categorical, FEATS_CATEGORICALS),
        ('feat_calendario', 'passthrough', FEATS_CALENDARIO)
    ],
    verbose_feature_names_out= False,
    remainder= 'drop'
)

model_1 = Pipeline(
    steps= [
        ('preprocessor', transform_mod_2),
        ('model', LogisticRegression(
            max_iter= 1500,
            random_state= CONFIG['random_state']
        ))
    ]
)
model_1.fit(X= X_train_fe, y= y_train_fe)
predict_proba = model_1.predict_proba(X= X_train_fe)[:,1]
roc_score = roc_auc_score(y_true= y_train_fe, y_score= predict_proba)
roc_score

0.5473182908488005

Modelo 2: Negocio + LAGs

In [22]:
FEATS_LAGS = [
    feat for feat in X_train_fe.columns
    if feat.endswith('_lag_1')
    or feat.endswith('_lag_3')
    or feat.endswith('_lag_6')
    or feat.endswith('_lag_12')
]
FEATS_FULL_MODEL_2 = FEATS_LAGS + FEATS_NEGOCIO
pipe_mod_2 = Pipeline(
    steps= [
        ('simple_imputer', SimpleImputer(strategy= 'constant', fill_value= 0)),
        ('standar_scaler', StandardScaler())
    ]
)
pipe_categorical = Pipeline(
    steps= [
        ('simple_imputer', SimpleImputer(strategy= 'constant', fill_value= 'MISSING')),
        ('one_hot_encoder', OneHotEncoder(handle_unknown= 'ignore'))
    ]
)

transform_mod_2 = ColumnTransformer(
    transformers= [
        ('feat_num', pipe_mod_2, FEATS_FULL_MODEL_2),
        ('feat_cat', pipe_categorical, FEATS_CATEGORICALS),
        ('feat_calendario', 'passthrough', FEATS_CALENDARIO)
    ],
    verbose_feature_names_out= False,
    remainder= 'drop'
)

model_2 = Pipeline(
    steps= [
        ('preprocessor', transform_mod_2),
        ('model', LogisticRegression(
            max_iter= 1500,
            random_state= CONFIG['random_state']
        ))
    ]
)
model_2.fit(X= X_train_fe, y= y_train_fe)
predict_proba = model_2.predict_proba(X= X_train_fe)[:,1]
roc_score = roc_auc_score(y_true= y_train_fe, y_score= predict_proba)
roc_score

0.5585268136130992

Modelo 3: Negocio + LAGs + rolling

In [23]:
FEATS_LAGS = [
    feat for feat in X_train_fe.columns
    if feat.endswith('_lag_1')
    or feat.endswith('_lag_3')
    or feat.endswith('_lag_6')
    or feat.endswith('_lag_12')
]
FEATS_ROLLING = [
    feat for feat in X_train_fe.columns
    if feat.endswith(r'_mean_3|_mean_6|_mean_12')
    or feat.endswith(r'_std_3|_std_6|_std_12')
    or feat.endswith(r'_max_6|_max_12')
    or feat.endswith(r'_min_6|_min_12')
    or feat.endswith(r'_sum_6|_sum_12')
]
FEATS_FULL_MODEL_2 = FEATS_LAGS + FEATS_NEGOCIO + FEATS_ROLLING
pipe_mod_2 = Pipeline(
    steps= [
        ('simple_imputer', SimpleImputer(strategy= 'constant', fill_value= 0)),
        ('standar_scaler', StandardScaler())
    ]
)
pipe_categorical = Pipeline(
    steps= [
        ('simple_imputer', SimpleImputer(strategy= 'constant', fill_value= 'MISSING')),
        ('one_hot_encoder', OneHotEncoder(handle_unknown= 'ignore'))
    ]
)

transform_mod_2 = ColumnTransformer(
    transformers= [
        ('feat_num', pipe_mod_2, FEATS_FULL_MODEL_2),
        ('feat_cat', pipe_categorical, FEATS_CATEGORICALS),
        ('feat_calendario', 'passthrough', FEATS_CALENDARIO)
    ],
    verbose_feature_names_out= False,
    remainder= 'drop'
)

model_3 = Pipeline(
    steps= [
        ('preprocessor', transform_mod_2),
        ('model', LogisticRegression(
            max_iter= 1500,
            random_state= CONFIG['random_state']
        ))
    ]
)
model_3.fit(X= X_train_fe, y= y_train_fe)
predict_proba = model_3.predict_proba(X= X_train_fe)[:,1]
roc_score = roc_auc_score(y_true= y_train_fe, y_score= predict_proba)
roc_score

0.5585268136130992

Modelo 4: Todo

LigthGBM

In [27]:
pipe_lgbm = Pipeline(
    steps= [
        ('preprocessor', preprocessor_b),
        ('model', LGBMClassifier())
    ]
)

pipe_lgbm.fit(
    X_train_fe,
    y_train_fe
)

pred = pipe_lgbm.predict_proba(
    X_train_fe
)[:,1]

roc_auc_score(
    y_train_fe,
    pred
)

[LightGBM] [Info] Number of positive: 20975, number of negative: 219025
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015474 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15678
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 158
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.087396 -> initscore=-2.345855
[LightGBM] [Info] Start training from score -2.345855


/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.6799683049771315

In [73]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipe_lgbm,
    X_train_fe,
    y_train_fe,
    scoring='roc_auc',
    cv=cv
)

scores.mean(), scores.std()

[LightGBM] [Info] Number of positive: 16780, number of negative: 175220
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011032 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15668
[LightGBM] [Info] Number of data points in the train set: 192000, number of used features: 158
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.087396 -> initscore=-2.345855
[LightGBM] [Info] Start training from score -2.345855


/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 16780, number of negative: 175220
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011882 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15697
[LightGBM] [Info] Number of data points in the train set: 192000, number of used features: 158
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.087396 -> initscore=-2.345855
[LightGBM] [Info] Start training from score -2.345855


/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 16780, number of negative: 175220
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011943 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15640
[LightGBM] [Info] Number of data points in the train set: 192000, number of used features: 158
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.087396 -> initscore=-2.345855
[LightGBM] [Info] Start training from score -2.345855


...

[LightGBM] [Info] Number of positive: 16780, number of negative: 175220
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012170 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15693
[LightGBM] [Info] Number of data points in the train set: 192000, number of used features: 158
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.087396 -> initscore=-2.345855
[LightGBM] [Info] Start training from score -2.345855


/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 16780, number of negative: 175220
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010255 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15667
[LightGBM] [Info] Number of data points in the train set: 192000, number of used features: 158
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.087396 -> initscore=-2.345855
[LightGBM] [Info] Start training from score -2.345855


/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


(np.float64(0.5488405378751506), np.float64(0.004888548147599986))

### Probando solo OOT

In [25]:
X_test_fe = test_fe.drop([TARGET], axis= 1)
y_test_fe = test_fe[TARGET]

In [28]:
proba_test_lgbm = pipe_lgbm.predict_proba(X=X_test_fe)[:,1]
roc_auc_score(y_true= y_test_fe, y_score= proba_test_lgbm)

/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.5520918100543581